# HiOCFD 기준 데이터와 Re=1600 소산율 비교 (수정판)

**사용법:** 런타임 → 모두 실행. `results.pkl`이 없으면 업로드 창이 뜹니다. 끝나면 출력 전체를 복사해서 보내 주세요.

수정 사항: 비교를 본 연구의 기록 시각(간격 0.1)에서 하도록 바꿨고(기준 곡선이 더 촘촘하므로 보간 오차가 사라집니다), 소산율 열이 −dE/dt와 같은지 확인하며, 그림의 축 이름과 범례를 논문 표기로 고쳤습니다.

In [ ]:
# ===== 셀 1: results.pkl 올리기 + 기준 파일 받기 =====
import os, pickle, subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

if not os.path.exists("results.pkl"):
    from google.colab import files
    print("results.pkl을 선택해서 올려 주세요 (taylor_green_final에서 받은 파일)")
    files.upload()
results = pickle.load(open("results.pkl", "rb"))
print("불러온 결과:", [(r["N"], int(r["Re"])) for r in results])

REF = "spectral_Re1600_512.gdiag"
URL = "https://cfd.ku.edu/hiocfd/spectral_Re1600_512.gdiag"
if not os.path.exists(REF):
    subprocess.run(["wget", "-q", "-O", REF, URL])
print("기준 파일 크기:", os.path.getsize(REF) if os.path.exists(REF) else "없음", "바이트")
print("--- 파일 앞부분 ---")
print(open(REF, errors="ignore").read()[:700])


In [ ]:
# ===== 셀 2: 열 판별 + 비교(우리 기록 시각에서) + 그림 + 요약 =====
rows = []
for line in open(REF, errors="ignore"):
    s = line.strip()
    if not s or s.startswith("#"):
        continue
    try:
        vals = [float(x) for x in s.replace(",", " ").split()]
    except ValueError:
        continue
    if len(vals) >= 3:
        rows.append(vals)
w = min(len(r) for r in rows)
ref = np.array([r[:w] for r in rows])
t_ref = ref[:, 0]
print(f"기준 데이터: {ref.shape[0]}행 x {ref.shape[1]}열, t = {t_ref.min():g} ~ {t_ref.max():g}")

LOG = []
def P(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

cols = list(range(1, ref.shape[1]))
E_cols = [c for c in cols if abs(ref[0, c] - 0.125) < 0.01]
eps_cols = [c for c in cols if 0.011 < ref[:, c].max() < 0.015]
P("운동에너지로 보이는 열:", E_cols, "| 소산율로 보이는 열:", eps_cols)
if not eps_cols:
    raise SystemExit("소산율 열을 자동으로 못 찾았습니다. 위에 출력된 파일 앞부분을 보내 주세요.")
c = eps_cols[0]
e_ref = ref[:, c]
eps_ref_max = e_ref.max(); i_ref = int(np.argmax(e_ref))
P(f"\nε_ref,max (분모) = {eps_ref_max:.5f} at t = {t_ref[i_ref]:.3f}  (기준 열 전체 t = {t_ref.min():g}–{t_ref.max():g}의 최댓값)")

# 소산율 열이 -dE/dt와 같은지 확인
if E_cols:
    dEdt = -np.gradient(ref[:, E_cols[0]], t_ref)
    P(f"열 {c} 대 -dE/dt: max|차이|/ε_ref,max = {np.max(np.abs(dEdt - e_ref)) / eps_ref_max:.2e}")

ours = {r["N"]: r for r in results if abs(r["Re"] - 1600) < 1e-9 and abs(r.get("dtmax", 0.01) - 0.01) < 1e-12}
refspl = CubicSpline(t_ref, e_ref)
tf = np.linspace(8.0, 10.0, 200001); tp_ref = tf[np.argmax(refspl(tf))]
P("\n" + "=" * 72)
P("비교는 본 연구의 기록 시각(간격 0.1, t ≤ 10)에서 하고, 기준 곡선을 그 시각으로 보간했습니다.")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(t_ref, e_ref, "k-", lw=2, label="Reference (512³)")
for N, r in sorted(ours.items()):
    m = r["t"] <= 10 + 1e-9
    tt = r["t"][m]; e_o = r["eps"][m]
    dev = (e_o - np.interp(tt, t_ref, e_ref)) / eps_ref_max
    ip = int(np.argmax(e_o)); csr = CubicSpline(tt, e_o); tpo = tf[np.argmax(csr(tf))]
    imax = int(np.argmax(np.abs(dev)))
    slope = np.interp(tt[imax], t_ref, np.gradient(e_ref, t_ref))
    P(f"\nN={N}")
    P(f"  피크: 기록값 {e_o[ip]:.5f} (t={tt[ip]:.1f}), 스플라인 {csr(tpo):.5f} (t={tpo:.3f}); 기준 {eps_ref_max:.5f} (t={tp_ref:.3f}); "
      f"상대차 {100*(e_o[ip]-eps_ref_max)/eps_ref_max:+.2f}% (기록), {100*(csr(tpo)-eps_ref_max)/eps_ref_max:+.2f}% (스플라인)")
    P(f"  전체 편차: max|dev| = {np.abs(dev).max():.2e} at t={tt[imax]:.1f} (dev={dev[imax]:+.2e}), rms = {np.sqrt((dev**2).mean()):.2e}")
    P(f"  t<6.5: max|dev| = {np.abs(dev[tt < 6.5]).max():.2e};  t>=9.5: max dev = {dev[tt >= 9.5].max():+.2e} at t={tt[tt >= 9.5][np.argmax(dev[tt >= 9.5])]:.1f}, t=10 dev = {dev[-1]:+.2e}")
    P(f"  최대 편차 위치에서 기준 곡선의 기울기 = {slope:.2e} -> 같은 크기의 편차를 만드는 시간 이동 = {dev[imax]*eps_ref_max/slope:+.3f}")
    if E_cols:
        dE = np.abs(r["E"][m] - np.interp(tt, t_ref, ref[:, E_cols[0]])).max()
        P(f"  운동에너지 E(t): max|ΔE| = {dE:.2e}  (E(0)=0.125의 {100*dE/0.125:.3f}%)")
    ax[0].plot(tt, e_o, "--", label=f"This work ({N}³)")
    ax[1].plot(tt, dev, label=f"{N}³")
ax[0].axvline(10, color="gray", ls=":", lw=.8, label="end of this work (t = 10)")
ax[0].set_xlabel("t"); ax[0].set_ylabel("Dissipation rate, ε"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$(\varepsilon-\varepsilon_{\mathrm{ref}})/\varepsilon_{\mathrm{ref,max}}$")
ax[1].axhline(0, color="k", lw=.5); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig8_benchmark.png", dpi=150); plt.show()
open("benchmark_summary.txt", "w", encoding="utf-8").write("\n".join(LOG))
try:
    from google.colab import files
    files.download("fig8_benchmark.png"); files.download("benchmark_summary.txt")
except Exception:
    pass
print("\n위 출력 전체를 그대로 복사해서 보내 주세요.")
